# A/B тестирование: анализ поведения пользователей e-commerce платформы

**Датасет:** [E-Commerce Users of a French C2C Fashion Store](https://www.kaggle.com/datasets/jmmvutu/ecommerce-users-of-a-french-c2c-fashion-store) (Kaggle)  
**Инструменты:** Python, Pandas, Plotly, SciPy, StatsModels  

---

## Описание проекта

Платформа представляет собой французский C2C fashion-маркетплейс (~99 000 пользователей).  
Цель - выявить факторы, коррелирующие с покупательской активностью (`productsBought`)  
и проверить соответствующие гипотезы статистическими методами.

**Шаги исследования:**
1. Приоритизация гипотез (ICE / RICE)
2. Разведочный анализ (EDA)
3. AA-тест - проверка валидности метода сплитования
4. AB-тесты с проверкой допущений и бутстреп верификацией
5. Выводы и ограничения

## 1. Импорт библиотек и загрузка данных

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.power import TTestIndPower
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itables import show
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('6M-0K-99K_users_dataset_public.csv')

print(f'Размер датасета: {df.shape[0]:,} пользователей, {df.shape[1]} признака')
show(df.head())

Размер датасета: 98,913 пользователей, 24 признака


Loading ITables v2.7.3 from the internet... (need help?)


### 1.1 Чистка датасета

In [2]:
# Проверка пропущенных значений
print('Пропущенные значения:')
print(df.isna().sum())

# Проверка дубликатов
print(f'\nДубликатов: {df.duplicated().sum()}')

# Проверка дубликатов по идентификатору
print(f'Уникальных пользователей: {df["identifierHash"].nunique()}')
print(f'Всего строк: {len(df)}')

Пропущенные значения:
identifierHash         0
type                   0
country                0
language               0
socialNbFollowers      0
socialNbFollows        0
socialProductsLiked    0
productsListed         0
productsSold           0
productsPassRate       0
productsWished         0
productsBought         0
gender                 0
civilityGenderId       0
civilityTitle          0
hasAnyApp              0
hasAndroidApp          0
hasIosApp              0
hasProfilePicture      0
daysSinceLastLogin     0
seniority              0
seniorityAsMonths      0
seniorityAsYears       0
countryCode            0
dtype: int64

Дубликатов: 0
Уникальных пользователей: 98913
Всего строк: 98913


**Результат проверки качества данных:**
- Пропущенных значений: 0
- Дубликатов: 0  
- Каждая строка соответствует уникальному пользователю (98 913 пользователей)

Датасет не требует очистки и готов к анализу.

## 2. Разведочный анализ данных (EDA)

Перед тестированием изучим структуру данных и целевую метрику `productsBought`.

In [3]:
show(df.describe().T.round(2))

Loading ITables v2.7.3 from the internet... (need help?)


In [4]:
fig = px.histogram(
    df, x='productsBought', nbins=100,
    title='Распределение целевой метрики productsBought',
    labels={'productsBought': 'Количество покупок', 'count': 'Пользователей'}
)
fig.update_layout(showlegend=False)
fig.show()

print(f"Медиана: {df['productsBought'].median()}")
print(f"Среднее: {df['productsBought'].mean():.4f}")
print(f"Std: {df['productsBought'].std():.4f}")
print(f"Доля пользователей с 0 покупок: {(df['productsBought'] == 0).mean():.1%}")

Медиана: 0.0
Среднее: 0.1719
Std: 2.3323
Доля пользователей с 0 покупок: 94.5%


**Наблюдения:**
- Распределение сильно скошено вправо: большинство пользователей совершили 0 покупок
- Медиана = 0, среднее >> медианы
- Наличие выбросов (max ~ 400) - стандартный t-тест может быть ненадёжен

**Решение:** использовать бакетный t-тест (тест Уэлча) с верификацией через бутстреп

## 3. Приоритизация гипотез (ICE / RICE)

Перед запуском тестов расставим приоритеты с помощью фреймворков ICE и RICE.

| Параметр | Описание |
|---|---|
| **Reach** | Охват — сколько пользователей затронет гипотеза (1–10) |
| **Impact** | Влияние на метрику (1–10) |
| **Confidence** | Уверенность в наличии эффекта (1–10) |
| **Effort** | Затраты на проверку (1–10, чем больше тем дороже) |

**ICE** = Impact × Confidence / Effort  
**RICE** = Reach × Impact × Confidence / Effort

In [5]:
hypotheses = pd.DataFrame({
    'Гипотеза': [
        'Фото профиля -> покупки',
        'Приложение -> покупки',
        'iOS vs Android',
        'Подписки -> покупки',
        'Вишлист -> покупки'
    ],
    'Reach':      [2, 7, 3, 4, 5],
    'Impact':     [5, 8, 4, 7, 8],
    'Confidence': [6, 7, 5, 6, 7],
    'Effort':     [2, 4, 2, 3, 3]
})

hypotheses['ICE']  = (hypotheses['Impact'] * hypotheses['Confidence'] / hypotheses['Effort']).round(1)
hypotheses['RICE'] = (hypotheses['Reach'] * hypotheses['Impact'] * hypotheses['Confidence'] / hypotheses['Effort']).round(1)

show(hypotheses.sort_values('RICE', ascending=False))

Loading ITables v2.7.3 from the internet... (need help?)


In [6]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Приоритизация по ICE', 'Приоритизация по RICE'))

ice_sorted  = hypotheses.sort_values('ICE', ascending=True)
rice_sorted = hypotheses.sort_values('RICE', ascending=True)

fig.add_trace(go.Bar(x=ice_sorted['ICE'],  y=ice_sorted['Гипотеза'],  orientation='h', marker_color='steelblue', name='ICE'),  row=1, col=1)
fig.add_trace(go.Bar(x=rice_sorted['RICE'], y=rice_sorted['Гипотеза'], orientation='h', marker_color='salmon',    name='RICE'), row=1, col=2)

fig.update_layout(height=350, showlegend=False, title='Приоритизация гипотез')
fig.show()

**Вывод:**  
ICE и RICE дают схожий порядок приоритетов, но расходятся по фото профиля:  
- По **ICE** (без учёта охвата) - фото на 2-м месте  
- По **RICE** (с учётом охвата) - фото опускается вниз, т.к. охват мал (Reach=2)

Наиболее приоритетны: **вишлист** и **приложение** - высокий Impact при умеренном Effort.

## 4. AA-тест - проверка валидности сплитования

Прежде чем запускать AB-тесты, убедимся что механизм случайного разбиения работает корректно.  
Разделим всех пользователей случайно на две равные части и проверим что метрика не отличается.

**H₀:** случайный сплит не создаёт смещения  
**H₁:** сплит создаёт систематическое смещение

In [7]:
np.random.seed(42)

all_users = df['productsBought'].sample(frac=1, random_state=42).reset_index(drop=True)
mid = len(all_users) // 2
aa_group1 = all_users[:mid]
aa_group2 = all_users[mid:]

n_buckets = 50
aa_buckets1 = pd.Series([aa_group1.iloc[i::n_buckets].mean() for i in range(n_buckets)])
aa_buckets2 = pd.Series([aa_group2.iloc[i::n_buckets].mean() for i in range(n_buckets)])

t_stat, p_value = stats.ttest_ind(aa_buckets1, aa_buckets2)

print(f'Среднее группа 1: {aa_group1.mean():.4f}')
print(f'Среднее группа 2: {aa_group2.mean():.4f}')
print(f't-статистика:     {t_stat:.4f}')
print(f'p-value:          {p_value:.4f}')
print()
if p_value >= 0.05:
    print('OK AA-тест пройден - случайный сплит не создаёт смещения, метод валиден')
else:
    print('NOT OK AA-тест провален - сплит некорректен!')

Среднее группа 1: 0.1799
Среднее группа 2: 0.1639
t-статистика:     1.0598
p-value:          0.2918

OK AA-тест пройден - случайный сплит не создаёт смещения, метод валиден


## 5. AB-тесты

### Методология

**Бакетный t-тест:** пользователи разбиваются на бакеты, сравниваются бакетные средние.  
Бакетирование приближает распределение средних к нормальному (ЦПТ), снижает влияние выбросов.

**Тест Уэлча** (`equal_var=False`): используется вместо стандартного t-теста.  
Обоснование: во всех гипотезах группы сильно несбалансированы по размеру,  
а тест Левена фиксирует неравенство дисперсий. При таком сочетании  
стандартный t-тест завышает ошибку 1-го рода, тест Уэлча корректно  
её контролирует.

**Бутстреп верификация:** 1000 итераций с подвыборками равного размера.  
Если доля значимых тестов > 95% - результат считается устойчивым.

**Принятие решений по результатам теста:**  
В production-среде решение о раскатке фичи принимается на основе двух критериев:  
1. Статистически значимое отклонение от нуля (p < α)  
2. Статистически значимое отклонение от MDE - минимального эффекта,  
   интересного бизнесу (задаётся до теста совместно с продактом)

В данном проекте используются исторические данные без реального бизнес-контекста,  
поэтому MDE не задавался. Решения принимаются только на основе статистической  
значимости (p < 0.05) и верификации через бутстреп.

**Уровень значимости:** α = 0.05

In [8]:
def run_full_test(group_a, group_b, name_a, name_b, h0, h1, n_buckets=50, n_iterations=1000):
    print(f'Гипотеза: {name_a} vs {name_b}')
    print(f'H₀: {h0}')
    print(f'H₁: {h1}')
    
    buckets_a = pd.Series([group_a[i::n_buckets].mean() for i in range(n_buckets)])
    buckets_b = pd.Series([group_b[i::n_buckets].mean() for i in range(n_buckets)])
    
    # Нормальность
    _, p_norm_a = stats.shapiro(buckets_a)
    _, p_norm_b = stats.shapiro(buckets_b)
    print(f'\n1. Нормальность (Шапиро-Уилк):')
    print(f'   {name_a}: p={p_norm_a:.4f} => {"OK" if p_norm_a > 0.05 else "NOT OK"}')
    print(f'   {name_b}: p={p_norm_b:.4f} => {"OK" if p_norm_b > 0.05 else "NOT OK"}')
    
    # Равенство дисперсий
    _, p_lev = stats.levene(buckets_a, buckets_b)
    equal_var = p_lev > 0.05
    print(f'\n2. Равенство дисперсий (Левен): p={p_lev:.4f}')
    print(f'   => {"OK равны, стандартный t-тест" if equal_var else "NOT OK не равны, тест Уэлча"}')
    
    # T-тест
    t_stat, p_ttest = stats.ttest_ind(buckets_a, buckets_b, equal_var=equal_var)
    print(f'\n3. T-тест: t={t_stat:.4f}, p={p_ttest:.6f}')
    result = p_ttest < 0.05
    print(f'   => {"Отвергаем H₀ OK" if result else "Не можем отвергнуть H₀"}')
    
    # Бутстреп
    np.random.seed(42)
    n_sample = min(len(group_a), len(group_b))
    p_values = []
    for _ in range(n_iterations):
        sa = group_a.sample(n=n_sample, replace=False)
        sb = group_b.sample(n=n_sample, replace=False)
        _, p = stats.ttest_ind(sa, sb)
        p_values.append(p)
    p_values = pd.Series(p_values)
    sig_share = (p_values < 0.05).mean()
    print(f'\n4. Бутстреп ({n_iterations} итераций):')
    print(f'   Среднее p-value:          {p_values.mean():.6f}')
    print(f'   Доля значимых тестов:     {sig_share:.4f}')
    print(f'   => {"Результат устойчив OK" if sig_share > 0.95 else "Результат неустойчив NOT OK"}')
    
    # Размер выборки
    effect_size = abs(group_a.mean() - group_b.mean()) / np.sqrt((group_a.std()**2 + group_b.std()**2) / 2)
    analysis = TTestIndPower()
    n_required = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8)
    print(f'\n5. Размер выборки:')
    print(f'   Размер эффекта (Cohen\'s d): {effect_size:.4f}')
    print(f'   Необходимо:                 {n_required:.0f}')
    print(f'   Фактически (меньшая группа): {min(len(group_a), len(group_b)):,}')
    print(f'   => {"Данных достаточно OK" if min(len(group_a), len(group_b)) >= n_required else "Данных недостаточно NOT OK"}')
    
    print(f'\n6. Средние значения:')
    print(f'   {name_a}: {group_a.mean():.4f}')
    print(f'   {name_b}: {group_b.mean():.4f}')
    print(f'\n7. Selection bias: группы сформированы по самовыбору пользователей')
    print(f'   => результат показывает корреляцию, не причинно-следственную связь\n')

### 5.1 Фото профиля → покупки

In [9]:
group_a = df[df['hasProfilePicture'] == False]['productsBought'].reset_index(drop=True)
group_b = df[df['hasProfilePicture'] == True]['productsBought'].reset_index(drop=True)

run_full_test(
    group_a, group_b,
    'Без фото', 'С фото',
    h0='Пользователи с фото и без покупают одинаково',
    h1='Количество покупок отличается в зависимости от наличия фото'
)

Гипотеза: Без фото vs С фото
H₀: Пользователи с фото и без покупают одинаково
H₁: Количество покупок отличается в зависимости от наличия фото

1. Нормальность (Шапиро-Уилк):
   Без фото: p=0.0007 => NOT OK
   С фото: p=0.0000 => NOT OK

2. Равенство дисперсий (Левен): p=0.0000
   => NOT OK не равны, тест Уэлча

3. T-тест: t=10.3563, p=0.000000
   => Отвергаем H₀ OK

4. Бутстреп (1000 итераций):
   Среднее p-value:          0.000000
   Доля значимых тестов:     1.0000
   => Результат устойчив OK

5. Размер выборки:
   Размер эффекта (Cohen's d): 0.3146
   Необходимо:                 160
   Фактически (меньшая группа): 1,895
   => Данных достаточно OK

6. Средние значения:
   Без фото: 1.8375
   С фото: 0.1394

7. Selection bias: группы сформированы по самовыбору пользователей
   => результат показывает корреляцию, не причинно-следственную связь



**Интерпретация:**  
H₀ отвергнута, однако эффект **противоположен** ожидаемому - пользователи без фото покупают больше (1.84 vs 0.14).  
Вероятная причина: пользователи без фото — покупатели, пришедшие за товаром. Пользователи с фото - продавцы, заполняющие профиль для доверия, но сами покупающие меньше.

### 5.2 Наличие приложения → покупки

In [10]:
group_a = df[df['hasAnyApp'] == False]['productsBought'].reset_index(drop=True)
group_b = df[df['hasAnyApp'] == True]['productsBought'].reset_index(drop=True)

run_full_test(
    group_a, group_b,
    'Без приложения', 'С приложением',
    h0='Пользователи с приложением и без покупают одинаково',
    h1='Количество покупок отличается в зависимости от наличия приложения'
)

Гипотеза: Без приложения vs С приложением
H₀: Пользователи с приложением и без покупают одинаково
H₁: Количество покупок отличается в зависимости от наличия приложения

1. Нормальность (Шапиро-Уилк):
   Без приложения: p=0.0000 => NOT OK
   С приложением: p=0.0607 => OK

2. Равенство дисперсий (Левен): p=0.0000
   => NOT OK не равны, тест Уэлча

3. T-тест: t=-12.7789, p=0.000000
   => Отвергаем H₀ OK

4. Бутстреп (1000 итераций):
   Среднее p-value:          0.000000
   Доля значимых тестов:     1.0000
   => Результат устойчив OK

5. Размер выборки:
   Размер эффекта (Cohen's d): 0.0947
   Необходимо:                 1751
   Фактически (меньшая группа): 26,174
   => Данных достаточно OK

6. Средние значения:
   Без приложения: 0.1094
   С приложением: 0.3457

7. Selection bias: группы сформированы по самовыбору пользователей
   => результат показывает корреляцию, не причинно-следственную связь



**Интерпретация:**  
H₀ отвергнута. Пользователи с приложением покупают в ~3 раза больше (0.35 vs 0.11).  
Приложение — индикатор вовлечённости: активные пользователи и скачивают приложение, и покупают больше.  
Бизнес-инсайт: стимулирование установки приложения среди новых пользователей может повысить конверсию.

### 5.3 iOS vs Android

**Датасет:** только пользователи с приложением (`hasAnyApp == True`)

**Гипотезы:**  
H₀: платформа не влияет на покупательское поведение  
H₁: покупательское поведение отличается в зависимости от платформы

#### Шаг 1. Хи-квадрат - связь платформы с фактом покупки
*(метрика бинарная: купил / не купил)*

#### Шаг 2. Бакетный t-тест - различие в количестве покупок
*(метрика непрерывная: productsBought)*

In [11]:
from scipy.stats import chi2_contingency
import plotly.express as px

# Создаём бинарную метрику и фильтруем только пользователей с приложением
app_users = df[df['hasAnyApp'] == True].copy()
app_users['bought_any'] = app_users['productsBought'].apply(lambda x: 'Купил' if x > 0 else 'Не купил')
app_users['platform'] = app_users.apply(lambda x: 'iOS' if x['hasIosApp'] else 'Android', axis=1)

# Таблица сопряжённости
ct = pd.crosstab(app_users['bought_any'], app_users['platform'])
print('Таблица сопряжённости:')
show(ct)

# Визуализация
fig = px.histogram(
    app_users, x='bought_any', color='platform',
    barmode='group',
    histnorm='probability density',
    title='Доля купивших по платформам',
    labels={'bought_any': '', 'platform': 'Платформа'}
)
fig.show()

# Хи-квадрат
stat, p, dof, expected = chi2_contingency(ct)
print(f'\nстатистика Хи-квадрат: {stat:.4f}')
print(f'p-value: {p:.4f}')
print(f'Степени свободы: {dof}')
print()
if p < 0.05:
    print('Отвергаем H₀ - связь между платформой и фактом покупки есть')
else:
    print('Не можем отвергнуть H₀ - связи между платформой и фактом покупки нет')

Таблица сопряжённости:


Loading ITables v2.7.3 from the internet... (need help?)



статистика Хи-квадрат: 3.7715
p-value: 0.0521
Степени свободы: 1

Не можем отвергнуть H₀ - связи между платформой и фактом покупки нет


**Интерпретация:**  
H₀ не отвергнута (p=0.052). Связи между платформой и фактом покупки нет.  

Примечание: p-value близко к границе значимости (0.05).  
В реальной практике такой результат требует увеличения выборки  
или повторного теста — делать однозначный вывод преждевременно.

Проверим t-тест

In [12]:
group_a = df[df['hasIosApp'] == True]['productsBought'].reset_index(drop=True)
group_b = df[df['hasAndroidApp'] == True]['productsBought'].reset_index(drop=True)

run_full_test(
    group_a, group_b,
    'iOS', 'Android',
    h0='Пользователи iOS и Android покупают одинаково',
    h1='Количество покупок отличается в зависимости от платформы'
)

Гипотеза: iOS vs Android
H₀: Пользователи iOS и Android покупают одинаково
H₁: Количество покупок отличается в зависимости от платформы

1. Нормальность (Шапиро-Уилк):
   iOS: p=0.0002 => NOT OK
   Android: p=0.0000 => NOT OK

2. Равенство дисперсий (Левен): p=0.0102
   => NOT OK не равны, тест Уэлча

3. T-тест: t=-0.2794, p=0.780846
   => Не можем отвергнуть H₀

4. Бутстреп (1000 итераций):
   Среднее p-value:          0.620462
   Доля значимых тестов:     0.0050
   => Результат неустойчив NOT OK

5. Размер выборки:
   Размер эффекта (Cohen's d): 0.0045
   Необходимо:                 770361
   Фактически (меньшая группа): 4,819
   => Данных недостаточно NOT OK

6. Средние значения:
   iOS: 0.3562
   Android: 0.3690

7. Selection bias: группы сформированы по самовыбору пользователей
   => результат показывает корреляцию, не причинно-следственную связь



**Интерпретация:**  
H₀ не отвергнута (p=0.78, бутстреп доля значимых = 0.005). Платформа не влияет на покупательское поведение.  
Важен сам факт наличия приложения, а не его тип.

### 5.4 Количество подписок => покупки

In [13]:
median_follows = df['socialNbFollows'].median()
group_a = df[df['socialNbFollows'] <= median_follows]['productsBought'].reset_index(drop=True)
group_b = df[df['socialNbFollows'] > median_follows]['productsBought'].reset_index(drop=True)

print(f'Медиана подписок: {median_follows}')
run_full_test(
    group_a, group_b,
    'Мало подписок', 'Много подписок',
    h0='Количество подписок не связано с покупками',
    h1='Количество покупок отличается в зависимости от числа подписок'
)

Медиана подписок: 8.0
Гипотеза: Мало подписок vs Много подписок
H₀: Количество подписок не связано с покупками
H₁: Количество покупок отличается в зависимости от числа подписок

1. Нормальность (Шапиро-Уилк):
   Мало подписок: p=0.0000 => NOT OK
   Много подписок: p=0.0006 => NOT OK

2. Равенство дисперсий (Левен): p=0.0000
   => NOT OK не равны, тест Уэлча

3. T-тест: t=-14.2018, p=0.000000
   => Отвергаем H₀ OK

4. Бутстреп (1000 итераций):
   Среднее p-value:          0.000000
   Доля значимых тестов:     1.0000
   => Результат устойчив OK

5. Размер выборки:
   Размер эффекта (Cohen's d): 0.3126
   Необходимо:                 162
   Фактически (меньшая группа): 3,881
   => Данных достаточно OK

6. Средние значения:
   Мало подписок: 0.0985
   Много подписок: 1.9693

7. Selection bias: группы сформированы по самовыбору пользователей
   => результат показывает корреляцию, не причинно-следственную связь



**Интерпретация:**  
H₀ отвергнута. Пользователи с большим числом подписок покупают в ~20 раз больше (1.97 vs 0.10).  
Подписки — сильный индикатор общей активности на платформе: вовлечённые пользователи и подписываются, и покупают.

### 5.5 Вишлист → покупки

In [14]:
median_wished = df['productsWished'].median()
group_a = df[df['productsWished'] <= median_wished]['productsBought'].reset_index(drop=True)
group_b = df[df['productsWished'] > median_wished]['productsBought'].reset_index(drop=True)

print(f'Медиана вишлиста: {median_wished}')
run_full_test(
    group_a, group_b,
    'Пустой вишлист', 'Есть вишлист',
    h0='Наличие товаров в вишлисте не связано с покупками',
    h1='Количество покупок отличается в зависимости от наличия вишлиста'
)

Медиана вишлиста: 0.0
Гипотеза: Пустой вишлист vs Есть вишлист
H₀: Наличие товаров в вишлисте не связано с покупками
H₁: Количество покупок отличается в зависимости от наличия вишлиста

1. Нормальность (Шапиро-Уилк):
   Пустой вишлист: p=0.0006 => NOT OK
   Есть вишлист: p=0.0000 => NOT OK

2. Равенство дисперсий (Левен): p=0.0000
   => NOT OK не равны, тест Уэлча

3. T-тест: t=-18.0464, p=0.000000
   => Отвергаем H₀ OK

4. Бутстреп (1000 итераций):
   Среднее p-value:          0.000000
   Доля значимых тестов:     1.0000
   => Результат устойчив OK

5. Размер выборки:
   Размер эффекта (Cohen's d): 0.2478
   Необходимо:                 257
   Фактически (меньшая группа): 9,301
   => Данных достаточно OK

6. Средние значения:
   Пустой вишлист: 0.0519
   Есть вишлист: 1.3281

7. Selection bias: группы сформированы по самовыбору пользователей
   => результат показывает корреляцию, не причинно-следственную связь



**Интерпретация:**  
H₀ отвергнута. Самый сильный эффект из всех тестов (t=-18). Пользователи с вишлистом покупают в ~25 раз больше.  
Вишлист — прямой индикатор покупательского намерения. Бизнес-инсайт: продвижение фичи вишлиста среди новых пользователей может значимо повлиять на конверсию.

## 6. Итоговая таблица результатов

In [15]:
summary = pd.DataFrame({
    'Гипотеза': [
        'Фото профиля -> покупки',
        'Приложение -> покупки',
        'iOS vs Android',
        'Подписки -> покупки',
        'Вишлист -> покупки'
    ],
    'Среднее A': [1.8375, 0.1094, 0.3562, 0.0985, 0.0519],
    'Среднее B': [0.1394, 0.3457, 0.3690, 1.9693, 1.3281],
    'p-value (t-тест)': ['<0.001', '<0.001', '0.781', '<0.001', '<0.001'],
    'Бутстреп (доля знач.)': ['1.000', '1.000', '0.005', '1.000', '1.000'],
    'Результат': [
        'H₀ отвергнута, эффект обратный',
        'H₀ отвергнута ✓',
        'H₀ не отвергнута',
        'H₀ отвергнута ✓',
        'H₀ отвергнута ✓'
    ]
})
show(summary)

Loading ITables v2.7.3 from the internet... (need help?)


## 7. Ограничения исследования

1. **Данные не из реального эксперимента** - датасет представляет срез пользователей в один момент времени, а не результат контролируемого эксперимента. Из этого вытекают все остальные ограничения.

2. **Selection bias**. Группы сформированы по признаку самовыбора пользователей, а не случайным назначением. Поэтому все результаты показывают **корреляцию**, а не причинно-следственную связь. Настоящий AB тест требует рандомного назначения пользователей в группы до воздействия.

3. **Несбалансированные группы** - в некоторых тестах группы сильно различаются по размеру (например, 1895 vs 97018 для фото профиля).

4. **Сетевой эффект** - пользователи платформы взаимодействуют друг с другом  
   (подписки, лайки, покупки у конкретных продавцов). Это означает что группы  
   потенциально влияют друг на друга, нарушая допущение о независимости наблюдений.  
   В production-среде при наличии сетевого эффекта простая рандомизация по  
   пользователям недостаточна. В данном проекте этот эффект не учитывался.

5. **SRM (Sample Ratio Mismatch) не проверялся** - данная проверка валидна  
   только при контролируемом эксперименте с заранее заданным соотношением групп.  
   В нашем случае группы сформированы по признаку самовыбора, поэтому  
   проверка SRM неприменима.

6. **Нормальность нарушена** - тест Шапиро-Уилка отвергает нормальность бакетных средних в большинстве групп. Однако результаты верифицированы бутстреп-методом, который не требует нормальности, оба метода дают согласованные выводы.